# Dusk risk pricing: SOL collateral / USDC debt

This notebook reproduces the illustrative 7-day expected-loss study. It keeps Dusk's utilization IRM out of the calculation so the output is a pure risk-price add-on. SOLUSDT is used as a return proxy for SOL/USDC.

## Method and definitions

The 15 sealed, stratified 12-hour score windows are bootstrapped into 50,000 seven-day paths under three explicit regime mixtures. Each path is revalued at 100%, 75%, and 50% liquidation depth using Dusk's default CPMM full-collateral exact-in value. Gross pool shortfall is priced as expected loss plus a 15% annual return on ES99 minus expected loss. Regime weights are policy scenarios, not estimated real-world frequencies.

In [1]:
from pathlib import Path
import csv
import json
from risk_pricing import DEFAULT_HISTORY, MarketConfig, run_analysis

NOTEBOOK_DIR = Path.cwd()
RESULTS_DIR = NOTEBOOK_DIR / 'results'
CONFIG = MarketConfig(path_count=50_000, seed=42)
summary = run_analysis(DEFAULT_HISTORY, RESULTS_DIR, CONFIG)
print('analysis status:', summary['status'])
print('history checksum:', summary['history_diagnostics']['sha256'])

analysis status: illustrative_not_calibrated
history checksum: e7e6078f81eab5151fbd39343d6b67ba7644dda185fbebc4f729c72ce717fa05


## Central illustrative result

In [2]:
central = summary['central_scenario']
print(json.dumps({
    'pair_risk_apr_bps': central['pair_risk_apr_bps'],
    'expected_loss_bps_of_ead_7d': central['expected_loss_bps_of_ead'],
    'probability_any_liquidation': central['probability_any_liquidation'],
    'probability_lender_shortfall': central['probability_lender_shortfall'],
    'marginal_high_ltv_fee_bps': central['marginal_high_ltv_fee_bps'],
}, indent=2))

{
  "pair_risk_apr_bps": 2443.7608551777,
  "expected_loss_bps_of_ead_7d": 45.0333578583604,
  "probability_any_liquidation": 0.73774,
  "probability_lender_shortfall": 0.27872,
  "marginal_high_ltv_fee_bps": 114.18623974310503
}


## Position allocation and cross-subsidy

In [3]:
for row in summary['central_position_pricing']:
    print(
        f"{row['position']:8s} | LTV {row['impact_ltv_pct']:4.0f}% | "
        f"liq {100 * row['liquidation_probability']:5.1f}% | "
        f"risk APR {100 * row['position_risk_apr']:6.2f}% | "
        f"pair-minus-position {row['cross_subsidy_bps']:8.1f} bps"
    )

Low LTV  | LTV   50% | liq   0.0% | risk APR   0.00% | pair-minus-position   2443.8 bps
Mid LTV  | LTV   65% | liq   1.3% | risk APR   1.04% | pair-minus-position   2339.6 bps
High LTV | LTV   80% | liq  73.8% | risk APR  58.72% | pair-minus-position  -3428.3 bps


## Scenario grid

In [4]:
with (RESULTS_DIR / 'scenario_results.csv').open() as handle:
    scenario_rows = list(csv.DictReader(handle))
for row in scenario_rows:
    print(
        f"{row['regime_mix']:18s} | depth {row['depth_label']:>4s} | "
        f"EL {float(row['expected_loss_bps_of_ead']):6.1f} bps/7d | "
        f"risk APR {float(row['pair_risk_apr_bps']) / 100:6.2f}%"
    )

mostly_calm        | depth 100% | EL    4.1 bps/7d | risk APR   2.48%
mostly_calm        | depth  75% | EL    7.0 bps/7d | risk APR   4.09%
mostly_calm        | depth  50% | EL   18.4 bps/7d | risk APR  10.23%
illustrative_base  | depth 100% | EL   31.1 bps/7d | risk APR  16.98%
illustrative_base  | depth  75% | EL   45.0 bps/7d | risk APR  24.44%
illustrative_base  | depth  50% | EL   90.2 bps/7d | risk APR  48.47%
stress_heavy       | depth 100% | EL   62.7 bps/7d | risk APR  33.96%
stress_heavy       | depth  75% | EL   85.9 bps/7d | risk APR  46.24%
stress_heavy       | depth  50% | EL  155.7 bps/7d | risk APR  83.00%


## Validation

In [5]:
validation = summary['validation']
assert validation['history_checksum_matches']
assert validation['history_has_15_segments']
assert validation['risk_apr_monotonic_with_depth_loss']
assert validation['risk_apr_monotonic_with_stress_weight']
assert validation['shapley_revenue_identity_pass']
assert validation['central_repeat_within_3pct_relative']
assert validation['all_losses_bounded_by_ead']
assert validation['all_checks_pass']
print(json.dumps(validation, indent=2))

{
  "history_checksum_matches": true,
  "history_has_15_segments": true,
  "history_has_five_segments_per_stratum": true,
  "max_impact_ltv_within_borrow_cap": true,
  "risk_apr_monotonic_with_depth_loss": true,
  "risk_apr_monotonic_with_stress_weight": true,
  "shapley_revenue_identity_max_abs_usdc": 1.4551915228366852e-11,
  "shapley_revenue_identity_pass": true,
  "central_repeat_pair_apr_difference_bps": 34.41301916036309,
  "central_repeat_pair_apr_relative_difference": 0.014081991323926303,
  "central_repeat_within_3pct_relative": true,
  "all_losses_bounded_by_ead": true,
  "all_checks_pass": true
}


## Interpretation boundary

The modeled shortfall is before any pre-funded USDC insurance draw. Dusk's auction transfers SOL to an external liquidator rather than swapping it through the internal AMM, so the joint CPMM unwind is a conservative shared-resale-depth proxy. Production calibration needs keeper fills, external venue depth, position repayment duration, and the funded insurance balance.